# IEEE-CIS Fraud Detection – Tree-Guided & Cost-Aware Ready Preprocessing Pipeline

Bản này sửa lỗi quan trọng trong bản trước:

- Không để `group_amt_mean/median` bị fill bằng 0 rồi chia ra giá trị hàng tỷ.
- Dùng global train statistics để fill group stats bị thiếu ở validation/test.
- Denominator của ratio được `clip(lower=1.0)`.
- Thêm quantile clipping theo train split trước khi scale.
- Clip numeric matrix sau scaling để tránh outlier cực đoan.
- Mặc định lưu dạng `.parquet` để giảm dung lượng ổ đĩa. Nếu máy thiếu `pyarrow/fastparquet`, tự fallback sang `.csv.gz`.

Pipeline phục vụ cho:
- CNN branch
- DeepFM categorical branch
- DeepFM numerical branch
- Gated MoE fusion

Bản này bổ sung:
- `review_meta_*` để đánh giá cost-aware / review-budget.
- `row_id` để đồng bộ prediction của LightGBM teacher với CNN/DeepFM/MoE student.
- `TransactionAmt_raw` để tính captured fraud amount / expected utility.



### Memory-fixed update
Bản này tránh lỗi `MemoryError` bằng cách **không tạo bản copy có label cho full wide feature table** (`full_train_labeled`, `full_val_labeled`, `full_internal_test_labeled`).  
Các bảng wide `full_*` vẫn được lưu riêng cho LightGBM/CatBoost teacher; nhãn và metadata được lưu ở `review_meta_*` và `y_*`.


### Memory-fixed v2 update
Bản v2 sửa lỗi `NameError: cnn_train is not defined` bằng cách khôi phục đoạn tạo `cnn_train`, `deepfm_*`, `full_*` trước khi gắn label.  
Đồng thời sửa metadata để dùng `full_train.shape` thay vì `full_train_labeled.shape`.


In [1]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import OrdinalEncoder, StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("Libraries loaded.")

Libraries loaded.


## 1. Configuration

In [2]:
RAW_DIR = Path(r"D:\project\data")
OUTPUT_DIR = Path(r"D:\project\data\merge_paper_ready_tree_cost")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_TRANS_PATH = RAW_DIR / "train_transaction.csv"
TRAIN_ID_PATH    = RAW_DIR / "train_identity.csv"
TEST_TRANS_PATH  = RAW_DIR / "test_transaction.csv"
TEST_ID_PATH     = RAW_DIR / "test_identity.csv"

TARGET_COL = "isFraud"
ID_COL = "TransactionID"
TIME_COL = "TransactionDT"

# Save settings
# "parquet" is recommended. If parquet engine is missing, the code will save .csv.gz instead.
SAVE_FORMAT = "parquet"   # "parquet" or "csv_gz"

# To save disk, you can keep this False unless you need Kaggle submission files immediately.
SAVE_OFFICIAL_TEST = False

# Clipping settings
PRE_SCALE_Q_LOW = 0.001
PRE_SCALE_Q_HIGH = 0.999
POST_SCALE_CLIP = 20.0

print("Output dir:", OUTPUT_DIR)

Output dir: D:\project\data\merge_paper_ready_tree_cost


## 2. Load raw data and merge

In [3]:
print("Loading raw data...")

train_trans = pd.read_csv(TRAIN_TRANS_PATH)
train_id    = pd.read_csv(TRAIN_ID_PATH)
test_trans  = pd.read_csv(TEST_TRANS_PATH)
test_id     = pd.read_csv(TEST_ID_PATH)

train_df = train_trans.merge(train_id, on=ID_COL, how="left")
official_test_df = test_trans.merge(test_id, on=ID_COL, how="left")

print("Merged train shape        :", train_df.shape)
print("Merged official test shape:", official_test_df.shape)
print("Fraud ratio:", train_df[TARGET_COL].mean())

Loading raw data...
Merged train shape        : (590540, 434)
Merged official test shape: (506691, 433)
Fraud ratio: 0.03499000914417313


## 3. Time-based split

In [4]:
def time_based_split(df, time_col=TIME_COL, train_ratio=0.70, val_ratio=0.15):
    if time_col not in df.columns:
        raise ValueError(f"{time_col} not found. Cannot perform time-based split.")

    df_sorted = df.sort_values(time_col).reset_index(drop=True)
    n = len(df_sorted)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_part = df_sorted.iloc[:train_end].copy()
    val_part = df_sorted.iloc[train_end:val_end].copy()
    test_part = df_sorted.iloc[val_end:].copy()

    return train_part, val_part, test_part

train_raw, val_raw, internal_test_raw = time_based_split(train_df)

print("train_raw        :", train_raw.shape, "fraud:", train_raw[TARGET_COL].mean())
print("val_raw          :", val_raw.shape, "fraud:", val_raw[TARGET_COL].mean())
print("internal_test_raw:", internal_test_raw.shape, "fraud:", internal_test_raw[TARGET_COL].mean())
print("official_test_raw:", official_test_df.shape)

assert train_raw[TIME_COL].max() <= val_raw[TIME_COL].min()
assert val_raw[TIME_COL].max() <= internal_test_raw[TIME_COL].min()
print("Time split check: OK")

train_raw        : (413378, 434) fraud: 0.03516878014795175
val_raw          : (88581, 434) fraud: 0.03434145019812375
internal_test_raw: (88581, 434) fraud: 0.03480430340592226
official_test_raw: (506691, 433)
Time split check: OK


## 4. Split X / y / ids

In [5]:
def split_xy(df, has_target=True):
    ids = df[[ID_COL]].copy() if ID_COL in df.columns else None

    if has_target:
        y = df[TARGET_COL].astype(int).copy()
        X = df.drop(columns=[TARGET_COL], errors="ignore").copy()
        return X, y, ids

    X = df.copy()
    return X, None, ids

X_train, y_train, train_ids = split_xy(train_raw, has_target=True)
X_val, y_val, val_ids = split_xy(val_raw, has_target=True)
X_internal_test, y_internal_test, internal_test_ids = split_xy(internal_test_raw, has_target=True)
X_official_test, _, official_test_ids = split_xy(official_test_df, has_target=False)

print(X_train.shape, X_val.shape, X_internal_test.shape, X_official_test.shape)

(413378, 433) (88581, 433) (88581, 433) (506691, 433)


## 4.1 Review-budget metadata

In [6]:
def make_review_meta(df: pd.DataFrame, split_name: str, has_target: bool = True) -> pd.DataFrame:
    meta = pd.DataFrame({
        "row_id": np.arange(len(df), dtype=np.int64),
        "split": split_name,
    })

    if ID_COL in df.columns:
        meta[ID_COL] = df[ID_COL].values

    if TIME_COL in df.columns:
        meta["TransactionDT_raw"] = pd.to_numeric(df[TIME_COL], errors="coerce").values

    if "TransactionAmt" in df.columns:
        amt = pd.to_numeric(df["TransactionAmt"], errors="coerce").fillna(0).astype("float32")
        meta["TransactionAmt_raw"] = amt.values
    else:
        meta["TransactionAmt_raw"] = 0.0

    if has_target:
        meta[TARGET_COL] = df[TARGET_COL].astype(int).values
        meta["fraud_value_raw"] = np.where(
            meta[TARGET_COL].values == 1,
            meta["TransactionAmt_raw"].values,
            0.0,
        ).astype("float32")
    else:
        meta[TARGET_COL] = np.nan
        meta["fraud_value_raw"] = np.nan

    meta["review_cost_unit"] = 1.0
    return meta

review_meta_train = make_review_meta(train_raw, "train", has_target=True)
review_meta_val = make_review_meta(val_raw, "val", has_target=True)
review_meta_internal_test = make_review_meta(internal_test_raw, "internal_test", has_target=True)
review_meta_official_test = make_review_meta(official_test_df, "official_test", has_target=False)

print(review_meta_train.head())
print("review_meta_train:", review_meta_train.shape)
print("review_meta_val:", review_meta_val.shape)
print("review_meta_internal_test:", review_meta_internal_test.shape)
print("review_meta_official_test:", review_meta_official_test.shape)

   row_id  split  TransactionID  TransactionDT_raw  TransactionAmt_raw  isFraud  fraud_value_raw  review_cost_unit
0       0  train        2987000              86400                68.5        0              0.0               1.0
1       1  train        2987001              86401                29.0        0              0.0               1.0
2       2  train        2987002              86469                59.0        0              0.0               1.0
3       3  train        2987003              86499                50.0        0              0.0               1.0
4       4  train        2987004              86506                50.0        0              0.0               1.0
review_meta_train: (413378, 8)
review_meta_val: (88581, 8)
review_meta_internal_test: (88581, 8)
review_meta_official_test: (506691, 8)


## 5. Unified feature engineering

In [7]:
def add_unified_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    def safe_str(col):
        if col in df.columns:
            return df[col].fillna("missing").astype(str)
        return pd.Series(["missing"] * len(df), index=df.index)

    # Amount features
    if "TransactionAmt" in df.columns:
        amt = pd.to_numeric(df["TransactionAmt"], errors="coerce")
        df["TransactionAmt_log1p"] = np.log1p(amt.clip(lower=0))
        df["TransactionAmt_decimal"] = ((amt - np.floor(amt)).fillna(-1)).astype("float32")
        df["TransactionAmt_round"] = amt.fillna(-1).round(0)
        df["TransactionAmt_is_zero"] = (amt.fillna(0) == 0).astype("int8")

    # Time features
    if TIME_COL in df.columns:
        dt = pd.to_numeric(df[TIME_COL], errors="coerce").fillna(0)
        df["TransactionHour"] = ((dt // 3600) % 24).astype("int16")
        df["TransactionDay"] = ((dt // (3600 * 24)) % 7).astype("int16")
        df["TransactionWeek"] = (dt // (3600 * 24 * 7)).astype("int16")

    # Email features
    if "P_emaildomain" in df.columns and "R_emaildomain" in df.columns:
        p_email = safe_str("P_emaildomain")
        r_email = safe_str("R_emaildomain")
        df["email_match"] = (p_email == r_email).astype("int8")
        df["email_pair"] = p_email + "_" + r_email

    # Product-device
    if "ProductCD" in df.columns and "DeviceType" in df.columns:
        df["product_device_combo"] = safe_str("ProductCD") + "_" + safe_str("DeviceType")

    # UID / relationship features
    if "card1" in df.columns and "card2" in df.columns:
        df["uid_card12"] = safe_str("card1") + "_" + safe_str("card2")

    if "card1" in df.columns and "addr1" in df.columns:
        df["uid_card_addr"] = safe_str("card1") + "_" + safe_str("addr1")
        df["card_addr"] = safe_str("card1") + "_" + safe_str("addr1")

    if "card1" in df.columns and "P_emaildomain" in df.columns:
        df["uid_card_email"] = safe_str("card1") + "_" + safe_str("P_emaildomain")

    if "card1" in df.columns and "card2" in df.columns and "addr1" in df.columns:
        df["uid_card2_addr"] = safe_str("card1") + "_" + safe_str("card2") + "_" + safe_str("addr1")

    if "DeviceInfo" in df.columns and "addr1" in df.columns:
        df["uid_device_addr"] = safe_str("DeviceInfo") + "_" + safe_str("addr1")

    if "card1" in df.columns and "DeviceInfo" in df.columns:
        df["uid_card_device"] = safe_str("card1") + "_" + safe_str("DeviceInfo")

    if "DeviceType" in df.columns and "ProductCD" in df.columns:
        df["device_product"] = safe_str("DeviceType") + "_" + safe_str("ProductCD")

    return df

X_train = add_unified_features(X_train)
X_val = add_unified_features(X_val)
X_internal_test = add_unified_features(X_internal_test)
X_official_test = add_unified_features(X_official_test)

print("Feature engineering done.")
print("X_train        :", X_train.shape)
print("X_val          :", X_val.shape)
print("X_internal_test:", X_internal_test.shape)
print("X_official_test:", X_official_test.shape)

Feature engineering done.
X_train        : (413378, 451)
X_val          : (88581, 451)
X_internal_test: (88581, 451)
X_official_test: (506691, 451)


## 6. Align columns

In [8]:
def align_to_train_columns(X_train, *others):
    train_cols = X_train.columns
    aligned = []
    for X in others:
        aligned.append(X.reindex(columns=train_cols, fill_value=np.nan))
    return (X_train, *aligned)

X_train, X_val, X_internal_test, X_official_test = align_to_train_columns(
    X_train, X_val, X_internal_test, X_official_test
)

print("Aligned shapes:")
print(X_train.shape, X_val.shape, X_internal_test.shape, X_official_test.shape)

Aligned shapes:
(413378, 451) (88581, 451) (88581, 451) (506691, 451)


## 7. Missing indicators before imputation

In [9]:
def add_missing_indicators_from_train(X_train, X_val, X_internal_test, X_official_test, threshold=0.01):
    missing_ratio = X_train.isna().mean()
    indicator_cols = missing_ratio[missing_ratio >= threshold].index.tolist()
    indicator_cols = [c for c in indicator_cols if c != ID_COL]

    for col in indicator_cols:
        new_col = f"{col}_isna"
        X_train[new_col] = X_train[col].isna().astype("int8")
        X_val[new_col] = X_val[col].isna().astype("int8")
        X_internal_test[new_col] = X_internal_test[col].isna().astype("int8")
        X_official_test[new_col] = X_official_test[col].isna().astype("int8")

    return X_train, X_val, X_internal_test, X_official_test, indicator_cols

X_train, X_val, X_internal_test, X_official_test, missing_indicator_cols = add_missing_indicators_from_train(
    X_train, X_val, X_internal_test, X_official_test, threshold=0.01
)

print("Missing indicators added:", len(missing_indicator_cols))
print("Sample:", missing_indicator_cols[:20])

C:\Users\ASUS\AppData\Local\Temp\ipykernel_21784\3714682507.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_official_test[new_col] = X_official_test[col].isna().astype("int8")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_21784\3714682507.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[new_col] = X_train[col].isna().astype("int8")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_21784\3714682507.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, 

Missing indicators added: 323
Sample: ['card2', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14']


C:\Users\ASUS\AppData\Local\Temp\ipykernel_21784\3714682507.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_internal_test[new_col] = X_internal_test[col].isna().astype("int8")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_21784\3714682507.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_official_test[new_col] = X_official_test[col].isna().astype("int8")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_21784\3714682507.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inse

## 8. Drop extremely high-missing columns after creating indicators

In [10]:
missing_ratio = X_train.isna().mean().sort_values(ascending=False)
high_missing_cols = missing_ratio[missing_ratio > 0.90].index.tolist()
high_missing_cols = [c for c in high_missing_cols if c != ID_COL]

X_train = X_train.drop(columns=high_missing_cols, errors="ignore")
X_val = X_val.drop(columns=high_missing_cols, errors="ignore")
X_internal_test = X_internal_test.drop(columns=high_missing_cols, errors="ignore")
X_official_test = X_official_test.drop(columns=high_missing_cols, errors="ignore")

print("Dropped high-missing columns:", len(high_missing_cols))
print("Sample:", high_missing_cols[:20])

Dropped high-missing columns: 12
Sample: ['id_24', 'id_25', 'id_08', 'id_07', 'id_26', 'id_21', 'id_27', 'id_23', 'id_22', 'D7', 'dist2', 'id_18']


## 9. Frequency encoding using train split only

In [11]:
candidate_freq_cols = [
    "card1", "card2", "card3", "card5",
    "addr1", "addr2",
    "P_emaildomain", "R_emaildomain",
    "ProductCD", "DeviceType", "DeviceInfo",
    "uid_card12", "uid_card_addr", "uid_card_email", "uid_card2_addr",
    "uid_device_addr", "uid_card_device",
    "email_pair", "card_addr", "device_product", "product_device_combo"
]

freq_cols = [c for c in candidate_freq_cols if c in X_train.columns]
freq_maps = {}

for col in freq_cols:
    train_series = X_train[col].fillna("missing").astype(str)
    freq_map = train_series.value_counts(dropna=False).to_dict()
    freq_maps[col] = freq_map

    X_train[f"{col}_freq"] = train_series.map(freq_map).fillna(0).astype("float32")
    X_val[f"{col}_freq"] = X_val[col].fillna("missing").astype(str).map(freq_map).fillna(0).astype("float32")
    X_internal_test[f"{col}_freq"] = X_internal_test[col].fillna("missing").astype(str).map(freq_map).fillna(0).astype("float32")
    X_official_test[f"{col}_freq"] = X_official_test[col].fillna("missing").astype(str).map(freq_map).fillna(0).astype("float32")

print("Frequency encoding completed for", len(freq_cols), "columns.")
print(freq_cols)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_21784\758690790.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[f"{col}_freq"] = train_series.map(freq_map).fillna(0).astype("float32")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_21784\758690790.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_val[f"{col}_freq"] = X_val[col].fillna("missing").astype(str).map(freq_map).fillna(0).astype("float32")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_21784\758690790.py:21: PerformanceWarning: DataFrame is highly fragmented.  This 

Frequency encoding completed for 21 columns.
['card1', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain', 'ProductCD', 'DeviceType', 'DeviceInfo', 'uid_card12', 'uid_card_addr', 'uid_card_email', 'uid_card2_addr', 'uid_device_addr', 'uid_card_device', 'email_pair', 'card_addr', 'device_product', 'product_device_combo']


C:\Users\ASUS\AppData\Local\Temp\ipykernel_21784\758690790.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_official_test[f"{col}_freq"] = X_official_test[col].fillna("missing").astype(str).map(freq_map).fillna(0).astype("float32")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_21784\758690790.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[f"{col}_freq"] = train_series.map(freq_map).fillna(0).astype("float32")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_21784\758690790.py:20: PerformanceWarning: DataFrame is highl

## 10. Scale-safe group-based TransactionAmt statistics

Phần này là chỗ đã sửa lỗi nổ giá trị.

Nguyên tắc:
- Group statistics fit bằng train only.
- Nếu group ở val/test không có trong train, fill bằng global train mean/median/std.
- Ratio denominator được clip lower=1.0.
- Z-score denominator được clip lower=1.0.

In [12]:
candidate_group_cols = [
    "card1", "addr1",
    "uid_card12", "uid_card_addr", "uid_card_email", "uid_card2_addr",
    "uid_card_device", "uid_device_addr", "email_pair"
]

group_cols = [c for c in candidate_group_cols if c in X_train.columns]
group_amount_maps = {}

global_amt_mean = float(pd.to_numeric(X_train["TransactionAmt"], errors="coerce").mean()) if "TransactionAmt" in X_train.columns else 0.0
global_amt_median = float(pd.to_numeric(X_train["TransactionAmt"], errors="coerce").median()) if "TransactionAmt" in X_train.columns else 0.0
global_amt_std = float(pd.to_numeric(X_train["TransactionAmt"], errors="coerce").std()) if "TransactionAmt" in X_train.columns else 1.0
if not np.isfinite(global_amt_std) or global_amt_std <= 0:
    global_amt_std = 1.0

def apply_group_amount_features_scale_safe(X, col, stat_df):
    X = X.merge(stat_df, on=col, how="left")

    mean_col = f"{col}_amt_mean"
    std_col = f"{col}_amt_std"
    median_col = f"{col}_amt_median"

    X[mean_col] = pd.to_numeric(X[mean_col], errors="coerce").fillna(global_amt_mean)
    X[median_col] = pd.to_numeric(X[median_col], errors="coerce").fillna(global_amt_median)
    X[std_col] = pd.to_numeric(X[std_col], errors="coerce").fillna(global_amt_std)

    if "TransactionAmt" in X.columns:
        amt = pd.to_numeric(X["TransactionAmt"], errors="coerce").fillna(global_amt_median)

        denom_mean = X[mean_col].clip(lower=1.0)
        denom_median = X[median_col].clip(lower=1.0)
        denom_std = X[std_col].clip(lower=1.0)

        X[f"{col}_amt_ratio_mean"] = (amt / denom_mean).astype("float32")
        X[f"{col}_amt_ratio_median"] = (amt / denom_median).astype("float32")
        X[f"{col}_amt_zscore"] = ((amt - X[mean_col]) / denom_std).astype("float32")

    return X

if "TransactionAmt" in X_train.columns:
    for col in group_cols:
        stat_df = (
            X_train
            .groupby(col, dropna=False)["TransactionAmt"]
            .agg(["mean", "std", "median"])
            .reset_index()
        )
        stat_df.columns = [col, f"{col}_amt_mean", f"{col}_amt_std", f"{col}_amt_median"]

        stat_df[f"{col}_amt_mean"] = pd.to_numeric(stat_df[f"{col}_amt_mean"], errors="coerce").fillna(global_amt_mean)
        stat_df[f"{col}_amt_median"] = pd.to_numeric(stat_df[f"{col}_amt_median"], errors="coerce").fillna(global_amt_median)
        stat_df[f"{col}_amt_std"] = pd.to_numeric(stat_df[f"{col}_amt_std"], errors="coerce").fillna(global_amt_std)
        stat_df[f"{col}_amt_std"] = stat_df[f"{col}_amt_std"].clip(lower=1.0)

        group_amount_maps[col] = stat_df

        X_train = apply_group_amount_features_scale_safe(X_train, col, stat_df)
        X_val = apply_group_amount_features_scale_safe(X_val, col, stat_df)
        X_internal_test = apply_group_amount_features_scale_safe(X_internal_test, col, stat_df)
        X_official_test = apply_group_amount_features_scale_safe(X_official_test, col, stat_df)

print("Scale-safe group-based amount features completed for", len(group_cols), "columns.")
print(group_cols)
print("Global amount stats:", global_amt_mean, global_amt_median, global_amt_std)
print("Shapes:", X_train.shape, X_val.shape, X_internal_test.shape, X_official_test.shape)

Scale-safe group-based amount features completed for 9 columns.
['card1', 'addr1', 'uid_card12', 'uid_card_addr', 'uid_card_email', 'uid_card2_addr', 'uid_card_device', 'uid_device_addr', 'email_pair']
Global amount stats: 134.60489153994646 68.95 238.00769000173503
Shapes: (413378, 837) (88581, 837) (88581, 837) (506691, 837)


## 11. Remove ID and define categorical/numerical columns

In [13]:
for X in [X_train, X_val, X_internal_test, X_official_test]:
    if ID_COL in X.columns:
        X.drop(columns=[ID_COL], inplace=True)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

print("Categorical columns:", len(cat_cols))
print("Numerical columns  :", len(num_cols))
print("Sample categorical:", cat_cols[:30])

Categorical columns: 39
Numerical columns  : 797
Sample categorical: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo', 'email_pair']


## 12. Ordinal encode categorical fields for DeepFM

In [14]:
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

if len(cat_cols) > 0:
    X_train_cat = pd.DataFrame(
        encoder.fit_transform(X_train[cat_cols].fillna("missing").astype(str)),
        columns=cat_cols,
        index=X_train.index,
    ).astype("int64")

    X_val_cat = pd.DataFrame(
        encoder.transform(X_val[cat_cols].fillna("missing").astype(str)),
        columns=cat_cols,
        index=X_val.index,
    ).astype("int64")

    X_internal_test_cat = pd.DataFrame(
        encoder.transform(X_internal_test[cat_cols].fillna("missing").astype(str)),
        columns=cat_cols,
        index=X_internal_test.index,
    ).astype("int64")

    X_official_test_cat = pd.DataFrame(
        encoder.transform(X_official_test[cat_cols].fillna("missing").astype(str)),
        columns=cat_cols,
        index=X_official_test.index,
    ).astype("int64")
else:
    X_train_cat = pd.DataFrame(index=X_train.index)
    X_val_cat = pd.DataFrame(index=X_val.index)
    X_internal_test_cat = pd.DataFrame(index=X_internal_test.index)
    X_official_test_cat = pd.DataFrame(index=X_official_test.index)

print("Categorical encoding done.")
print(X_train_cat.shape, X_val_cat.shape, X_internal_test_cat.shape, X_official_test_cat.shape)

Categorical encoding done.
(413378, 39) (88581, 39) (88581, 39) (506691, 39)


## 13. Numeric imputation, quantile clipping, scaling, post-scale clipping

Đây là đoạn quan trọng để audit không còn `max_abs_value` hàng tỷ.

In [15]:
X_train_num = X_train[num_cols].apply(pd.to_numeric, errors="coerce")
X_val_num = X_val[num_cols].apply(pd.to_numeric, errors="coerce")
X_internal_test_num = X_internal_test[num_cols].apply(pd.to_numeric, errors="coerce")
X_official_test_num = X_official_test[num_cols].apply(pd.to_numeric, errors="coerce")

# Replace inf before median
for X_num in [X_train_num, X_val_num, X_internal_test_num, X_official_test_num]:
    X_num.replace([np.inf, -np.inf], np.nan, inplace=True)

train_medians = X_train_num.median(numeric_only=True)

X_train_num = X_train_num.fillna(train_medians).fillna(0)
X_val_num = X_val_num.fillna(train_medians).fillna(0)
X_internal_test_num = X_internal_test_num.fillna(train_medians).fillna(0)
X_official_test_num = X_official_test_num.fillna(train_medians).fillna(0)

# Quantile clipping based on train only
clip_low = X_train_num.quantile(PRE_SCALE_Q_LOW)
clip_high = X_train_num.quantile(PRE_SCALE_Q_HIGH)

# Avoid invalid bounds
clip_low = clip_low.replace([np.inf, -np.inf], np.nan).fillna(X_train_num.min())
clip_high = clip_high.replace([np.inf, -np.inf], np.nan).fillna(X_train_num.max())

for col in X_train_num.columns:
    low = clip_low[col]
    high = clip_high[col]
    if pd.isna(low) or pd.isna(high) or low > high:
        continue
    X_train_num[col] = X_train_num[col].clip(low, high)
    X_val_num[col] = X_val_num[col].clip(low, high)
    X_internal_test_num[col] = X_internal_test_num[col].clip(low, high)
    X_official_test_num[col] = X_official_test_num[col].clip(low, high)

scaler = StandardScaler()

X_train_num_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_num),
    columns=num_cols,
    index=X_train_num.index,
).astype("float32")

X_val_num_scaled = pd.DataFrame(
    scaler.transform(X_val_num),
    columns=num_cols,
    index=X_val_num.index,
).astype("float32")

X_internal_test_num_scaled = pd.DataFrame(
    scaler.transform(X_internal_test_num),
    columns=num_cols,
    index=X_internal_test_num.index,
).astype("float32")

X_official_test_num_scaled = pd.DataFrame(
    scaler.transform(X_official_test_num),
    columns=num_cols,
    index=X_official_test_num.index,
).astype("float32")

# Post-scale clipping for neural networks
for X_scaled in [X_train_num_scaled, X_val_num_scaled, X_internal_test_num_scaled, X_official_test_num_scaled]:
    X_scaled.clip(-POST_SCALE_CLIP, POST_SCALE_CLIP, inplace=True)
    X_scaled.replace([np.inf, -np.inf], 0, inplace=True)
    X_scaled.fillna(0, inplace=True)

clipping_bounds = {
    "low": clip_low.to_dict(),
    "high": clip_high.to_dict(),
    "pre_scale_q_low": PRE_SCALE_Q_LOW,
    "pre_scale_q_high": PRE_SCALE_Q_HIGH,
    "post_scale_clip": POST_SCALE_CLIP,
}

print("Numeric imputation + clipping + scaling done.")
print(X_train_num_scaled.shape, X_val_num_scaled.shape, X_internal_test_num_scaled.shape, X_official_test_num_scaled.shape)
print("Train mean abs:", float(np.mean(np.abs(X_train_num_scaled.to_numpy()))))
print("Val mean abs  :", float(np.mean(np.abs(X_val_num_scaled.to_numpy()))))
print("Test mean abs :", float(np.mean(np.abs(X_internal_test_num_scaled.to_numpy()))))
print("Official mean abs:", float(np.mean(np.abs(X_official_test_num_scaled.to_numpy()))))
print("Official max abs:", float(np.max(np.abs(X_official_test_num_scaled.to_numpy()))))

Numeric imputation + clipping + scaling done.
(413378, 797) (88581, 797) (88581, 797) (506691, 797)
Train mean abs: 0.5605720281600952
Val mean abs  : 0.5168551206588745
Test mean abs : 0.5677453279495239
Official mean abs: 0.548994243144989
Official max abs: 20.0


## 14. Build branch-specific outputs

In [16]:
# Build branch-specific outputs.
# Memory-safe rule:
# - Do not copy full_* only to append labels.
# - Keep full_* unlabeled and save y_* separately.

# CNN dense input
cnn_train = X_train_num_scaled
cnn_val = X_val_num_scaled
cnn_internal_test = X_internal_test_num_scaled
cnn_official_test = X_official_test_num_scaled

# DeepFM inputs
deepfm_cat_train = X_train_cat
deepfm_cat_val = X_val_cat
deepfm_cat_internal_test = X_internal_test_cat
deepfm_cat_official_test = X_official_test_cat

deepfm_num_train = X_train_num_scaled
deepfm_num_val = X_val_num_scaled
deepfm_num_internal_test = X_internal_test_num_scaled
deepfm_num_official_test = X_official_test_num_scaled

# Full processed features for tree-based teacher / ML baselines.
# Keep these UNLABELED to avoid huge MemoryError.
full_train = pd.concat([X_train_num_scaled, X_train_cat], axis=1)
full_val = pd.concat([X_val_num_scaled, X_val_cat], axis=1)
full_internal_test = pd.concat([X_internal_test_num_scaled, X_internal_test_cat], axis=1)
full_official_test = pd.concat([X_official_test_num_scaled, X_official_test_cat], axis=1)

def add_label_small(X, y):
    """
    Use only for branch tables expected by existing DL scripts.
    Do not use this for full_train/full_val/full_internal_test.
    """
    out = X.copy()
    out[TARGET_COL] = y.values
    return out

cnn_train_labeled = add_label_small(cnn_train, y_train)
cnn_val_labeled = add_label_small(cnn_val, y_val)
cnn_internal_test_labeled = add_label_small(cnn_internal_test, y_internal_test)

deepfm_cat_train_labeled = add_label_small(deepfm_cat_train, y_train)
deepfm_cat_val_labeled = add_label_small(deepfm_cat_val, y_val)
deepfm_cat_internal_test_labeled = add_label_small(deepfm_cat_internal_test, y_internal_test)

deepfm_num_train_labeled = add_label_small(deepfm_num_train, y_train)
deepfm_num_val_labeled = add_label_small(deepfm_num_val, y_val)
deepfm_num_internal_test_labeled = add_label_small(deepfm_num_internal_test, y_internal_test)

print("CNN train:", cnn_train_labeled.shape)
print("DeepFM cat train:", deepfm_cat_train_labeled.shape)
print("DeepFM num train:", deepfm_num_train_labeled.shape)
print("Full train unlabeled:", full_train.shape)
print("Full val unlabeled:", full_val.shape)
print("Full internal test unlabeled:", full_internal_test.shape)

CNN train: (413378, 798)
DeepFM cat train: (413378, 40)
DeepFM num train: (413378, 798)
Full train unlabeled: (413378, 836)
Full val unlabeled: (88581, 836)
Full internal test unlabeled: (88581, 836)


## 15. Metadata and cardinalities

In [17]:
categorical_cardinalities = []

for col in cat_cols:
    if col in deepfm_cat_train.columns:
        max_code = int(deepfm_cat_train[col].max()) if len(deepfm_cat_train) > 0 else 0
        cardinality = max_code + 2  # unknown -1 will be shifted to 0 in PyTorch Dataset
        categorical_cardinalities.append(cardinality)

metadata = {
    "target_col": TARGET_COL,
    "id_col": ID_COL,
    "time_col": TIME_COL,
    "split_type": "time_based_70_15_15",
    "scale_fix": {
        "group_stats_fill": "global_train_amount_statistics",
        "ratio_denominator_clip_lower": 1.0,
        "pre_scale_q_low": PRE_SCALE_Q_LOW,
        "pre_scale_q_high": PRE_SCALE_Q_HIGH,
        "post_scale_clip": POST_SCALE_CLIP,
    },
    "save_format": SAVE_FORMAT,
    "save_official_test": SAVE_OFFICIAL_TEST,
    "cost_aware_ready": True,
    "teacher_distillation_ready": True,
    "review_metadata_files": [
        "review_meta_train",
        "review_meta_val",
        "review_meta_internal_test",
        "review_meta_official_test",
    ],
    "dropped_high_missing_cols": high_missing_cols,
    "missing_indicator_source_cols": missing_indicator_cols,
    "frequency_encoded_columns": freq_cols,
    "group_amount_columns": group_cols,
    "categorical_columns": cat_cols,
    "numerical_columns": num_cols,
    "categorical_cardinalities": categorical_cardinalities,
    "cnn_dense_dim": int(cnn_train.shape[1]),
    "deepfm_num_dim": int(deepfm_num_train.shape[1]),
    "deepfm_num_categorical": int(len(cat_cols)),
    "train_shape": list(full_train.shape),
    "val_shape": list(full_val.shape),
    "internal_test_shape": list(full_internal_test.shape),
    "official_test_shape": list(full_official_test.shape),
    "fraud_ratio_train": float(y_train.mean()),
    "fraud_ratio_val": float(y_val.mean()),
    "fraud_ratio_internal_test": float(y_internal_test.mean()),
}

print(json.dumps({
    "cnn_dense_dim": metadata["cnn_dense_dim"],
    "deepfm_num_dim": metadata["deepfm_num_dim"],
    "deepfm_num_categorical": metadata["deepfm_num_categorical"],
    "fraud_ratio_train": metadata["fraud_ratio_train"],
    "fraud_ratio_val": metadata["fraud_ratio_val"],
    "fraud_ratio_internal_test": metadata["fraud_ratio_internal_test"],
    "scale_fix": metadata["scale_fix"]
}, indent=2))

{
  "cnn_dense_dim": 797,
  "deepfm_num_dim": 797,
  "deepfm_num_categorical": 39,
  "fraud_ratio_train": 0.03516878014795175,
  "fraud_ratio_val": 0.03434145019812375,
  "fraud_ratio_internal_test": 0.03480430340592226,
  "scale_fix": {
    "group_stats_fill": "global_train_amount_statistics",
    "ratio_denominator_clip_lower": 1.0,
    "pre_scale_q_low": 0.001,
    "pre_scale_q_high": 0.999,
    "post_scale_clip": 20.0
  }
}


## 16. Save processed files

In [18]:
def save_table(df: pd.DataFrame, name: str):
    if SAVE_FORMAT == "parquet":
        path = OUTPUT_DIR / f"{name}.parquet"
        try:
            df.to_parquet(path, index=False)
            return path
        except Exception as e:
            print(f"[WARN] Parquet save failed for {name}: {e}")
            print("[WARN] Falling back to csv.gz")
            path = OUTPUT_DIR / f"{name}.csv.gz"
            df.to_csv(path, index=False, compression="gzip")
            return path

    if SAVE_FORMAT == "csv_gz":
        path = OUTPUT_DIR / f"{name}.csv.gz"
        df.to_csv(path, index=False, compression="gzip")
        return path

    raise ValueError("SAVE_FORMAT must be 'parquet' or 'csv_gz'.")

saved_paths = {}

# Essential training files
saved_paths["cnn_train"] = save_table(cnn_train_labeled, "cnn_train")
saved_paths["cnn_val"] = save_table(cnn_val_labeled, "cnn_val")
saved_paths["cnn_internal_test"] = save_table(cnn_internal_test_labeled, "cnn_internal_test")

saved_paths["deepfm_cat_train"] = save_table(deepfm_cat_train_labeled, "deepfm_cat_train")
saved_paths["deepfm_cat_val"] = save_table(deepfm_cat_val_labeled, "deepfm_cat_val")
saved_paths["deepfm_cat_internal_test"] = save_table(deepfm_cat_internal_test_labeled, "deepfm_cat_internal_test")

saved_paths["deepfm_num_train"] = save_table(deepfm_num_train_labeled, "deepfm_num_train")
saved_paths["deepfm_num_val"] = save_table(deepfm_num_val_labeled, "deepfm_num_val")
saved_paths["deepfm_num_internal_test"] = save_table(deepfm_num_internal_test_labeled, "deepfm_num_internal_test")

# Baseline files
saved_paths["full_train"] = save_table(full_train, "full_train")  # unlabeled, memory-safe
saved_paths["full_val"] = save_table(full_val, "full_val")  # unlabeled, memory-safe
saved_paths["full_internal_test"] = save_table(full_internal_test, "full_internal_test")  # unlabeled, memory-safe

# Review-budget / cost-aware metadata
saved_paths["review_meta_train"] = save_table(review_meta_train, "review_meta_train")
saved_paths["review_meta_val"] = save_table(review_meta_val, "review_meta_val")
saved_paths["review_meta_internal_test"] = save_table(review_meta_internal_test, "review_meta_internal_test")
saved_paths["review_meta_official_test"] = save_table(review_meta_official_test, "review_meta_official_test")

# Label-only files for memory-safe teacher/student training
saved_paths["y_train"] = save_table(pd.DataFrame({TARGET_COL: y_train.values}), "y_train")
saved_paths["y_val"] = save_table(pd.DataFrame({TARGET_COL: y_val.values}), "y_val")
saved_paths["y_internal_test"] = save_table(pd.DataFrame({TARGET_COL: y_internal_test.values}), "y_internal_test")

# Official test optional
if SAVE_OFFICIAL_TEST:
    saved_paths["cnn_official_test"] = save_table(cnn_official_test, "cnn_official_test")
    saved_paths["deepfm_cat_official_test"] = save_table(deepfm_cat_official_test, "deepfm_cat_official_test")
    saved_paths["deepfm_num_official_test"] = save_table(deepfm_num_official_test, "deepfm_num_official_test")
    saved_paths["full_official_test"] = save_table(full_official_test, "full_official_test")

print("Saved files:")
for k, v in saved_paths.items():
    print(k, "->", v)

Saved files:
cnn_train -> D:\project\data\merge_paper_ready_tree_cost\cnn_train.parquet
cnn_val -> D:\project\data\merge_paper_ready_tree_cost\cnn_val.parquet
cnn_internal_test -> D:\project\data\merge_paper_ready_tree_cost\cnn_internal_test.parquet
deepfm_cat_train -> D:\project\data\merge_paper_ready_tree_cost\deepfm_cat_train.parquet
deepfm_cat_val -> D:\project\data\merge_paper_ready_tree_cost\deepfm_cat_val.parquet
deepfm_cat_internal_test -> D:\project\data\merge_paper_ready_tree_cost\deepfm_cat_internal_test.parquet
deepfm_num_train -> D:\project\data\merge_paper_ready_tree_cost\deepfm_num_train.parquet
deepfm_num_val -> D:\project\data\merge_paper_ready_tree_cost\deepfm_num_val.parquet
deepfm_num_internal_test -> D:\project\data\merge_paper_ready_tree_cost\deepfm_num_internal_test.parquet
full_train -> D:\project\data\merge_paper_ready_tree_cost\full_train.parquet
full_val -> D:\project\data\merge_paper_ready_tree_cost\full_val.parquet
full_internal_test -> D:\project\data\merg

## Optional memory cleanup

In [19]:
# Optional memory cleanup after all outputs are saved.
import gc

to_delete = [
    "cnn_train", "cnn_val", "cnn_internal_test",
    "deepfm_cat_train", "deepfm_cat_val", "deepfm_cat_internal_test",
    "deepfm_num_train", "deepfm_num_val", "deepfm_num_internal_test",
    "full_train", "full_val", "full_internal_test",
]

for name in to_delete:
    if name in globals():
        del globals()[name]

gc.collect()
print("Memory cleanup completed.")

Memory cleanup completed.


In [20]:
# Save metadata and preprocessing objects
metadata_path = OUTPUT_DIR / "preprocessing_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "ordinal_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

with open(OUTPUT_DIR / "standard_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open(OUTPUT_DIR / "train_medians.pkl", "wb") as f:
    pickle.dump(train_medians.to_dict(), f)

with open(OUTPUT_DIR / "frequency_maps.pkl", "wb") as f:
    pickle.dump(freq_maps, f)

with open(OUTPUT_DIR / "group_amount_maps.pkl", "wb") as f:
    pickle.dump(group_amount_maps, f)

with open(OUTPUT_DIR / "clipping_bounds.pkl", "wb") as f:
    pickle.dump(clipping_bounds, f)

print("Saved metadata and preprocessing objects:")
print(metadata_path)
print(OUTPUT_DIR / "ordinal_encoder.pkl")
print(OUTPUT_DIR / "standard_scaler.pkl")
print(OUTPUT_DIR / "train_medians.pkl")
print(OUTPUT_DIR / "frequency_maps.pkl")
print(OUTPUT_DIR / "group_amount_maps.pkl")
print(OUTPUT_DIR / "clipping_bounds.pkl")

Saved metadata and preprocessing objects:
D:\project\data\merge_paper_ready_tree_cost\preprocessing_metadata.json
D:\project\data\merge_paper_ready_tree_cost\ordinal_encoder.pkl
D:\project\data\merge_paper_ready_tree_cost\standard_scaler.pkl
D:\project\data\merge_paper_ready_tree_cost\train_medians.pkl
D:\project\data\merge_paper_ready_tree_cost\frequency_maps.pkl
D:\project\data\merge_paper_ready_tree_cost\group_amount_maps.pkl
D:\project\data\merge_paper_ready_tree_cost\clipping_bounds.pkl


## 17. Expected audit result

Sau khi chạy file này, chạy lại `dataset_audit_ieee_cis_light.py`.

Kết quả tốt nên gần như:

```text
NaN = 0
Inf = 0
Object columns = 0
Column consistency = true
cnn_train mean_abs ≈ 0.5–2
cnn_val mean_abs ≈ 0.5–3
cnn_internal_test mean_abs ≈ 0.5–3
max_abs_value <= 20 hoặc quanh 20 vì đã post-scale clip
```

## Ghi chú dùng cho LightGBM/CatBoost teacher

Từ bản memory-fixed này:

```text
full_train / full_val / full_internal_test
```

là bảng feature **không có cột `isFraud`** để tránh copy dữ liệu lớn.

Khi train teacher, đọc nhãn từ:

```text
y_train
y_val
y_internal_test
```

hoặc từ:

```text
review_meta_train
review_meta_val
review_meta_internal_test
```

Ví dụ:

```python
X_train = read_table("full_train")
y_train = read_table("y_train")["isFraud"]
```